In [ ]:
from google.colab import drive
drive.mount("/content/gdrive", force_remount=True)

In [ ]:
import sys
import os
working_path = '???'
sys.path.append(working_path )
os.chdir(working_path )

In [ ]:
DATA_NAME = 'fruits/'
DATA_NAME_PATH = working_path +  'data/' + DATA_NAME
DATASET_PATH = DATA_NAME_PATH + 'dataset/'

class_list = ['apple_fruits', 'banana', 'coconut_fruits']
print(class_list)
class_len = len(class_list)
print(class_len)

BATCH_SIZE = 8

### Load Dataset (or Equivalent)

In [ ]:
# Images saved in folders with the following patterns
# Images finally saved in folders with the following patterns
# data_path/
#     train/
#        class0/
#           classs0_0.jpg
#           ...
#           classs0_k.jpg
#       class1/
#         classs1_0.jpg
#         ...
#         classs1_k.jpg
#       ...
#       classN/
#         classsN_0.jpg
#         ...
#         classsN_k.jpg
#     validation/
#        class0/
#        ...
#        classN/
#     test/
#        class0/
#        ...
#        classN
# npy/
#     X_train.npy
#     y_train.npy
#     X_test.npy
#     y_test.npy

In [ ]:
# Load numpy_files
import numpy as np

NUMPY_PATH = DATA_NAME_PATH + 'npy/'
print(NUMPY_PATH)

X_train = np.load(NUMPY_PATH + 'X_train.npy')
y_train = np.load(NUMPY_PATH + 'y_train.npy')

X_val = np.load(NUMPY_PATH + 'X_val.npy')
y_val = np.load(NUMPY_PATH + 'y_val.npy')

X_test = np.load(NUMPY_PATH + 'X_test.npy')
y_test = np.load(NUMPY_PATH + 'y_test.npy')

In [ ]:
X_train.shape

In [ ]:
IMG_SIZE = X_train.shape[1]
print(IMG_SIZE)

In [ ]:
# Display a few images and labels
import matplotlib.pyplot as plt
%matplotlib inline

class_names = np.array(class_list)

plt.figure(figsize=(15,10))
inx = np.random.choice(X_train.shape[0], 15, replace=False)
for n, i in enumerate(inx):
    ax = plt.subplot(3,5,n+1)
    plt.imshow(X_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis('off')

### Load Existing Deep Learning Model

In [ ]:
import tensorflow as tf

In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                                                include_top=False,
                                                weights="imagenet"
                                                )

In [ ]:
for layer in base_model.layers:
    print(layer.name, layer.output.shape)

In [ ]:
base_model.summary()

In [ ]:
from tensorflow.keras.models import  Sequential, Model, load_model

final_model = tf.keras.Sequential([base_model,
                                  tf.keras.layers.GlobalAveragePooling2D(),
                                  tf.keras.layers.BatchNormalization(),
                                  tf.keras.layers.Dense(128, activation='relu'),
                                  tf.keras.layers.Dropout(0.5),
                                  tf.keras.layers.Dense(class_len, activation='softmax')  # For multi-class
                            ])
# Notice that the last layer, the output size is equal to number of classes

In [ ]:
y_train

In [ ]:
# Compile Model

final_model.layers[0].trainable = False
final_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                    loss='sparse_categorical_crossentropy',  # classes = 0,1,2,3..'
                    #loss='categorical_crossentropy',        # classes = one hot
                    metrics = ["accuracy"]
)

In [ ]:
for layer in final_model.layers:
    print(layer.name, layer.output.shape)

In [ ]:
final_model.summary()

In [ ]:
# For resizing images that are not 224 x 224
#X_train_resized = tf.image.resize(X_train, IMG_SIZE)
#X_val_resized = tf.image.resize(X_val, IMG_SIZE)
#X_test_resized = tf.image.resize(X_test, IMG_SIZE)

#print(f"Resized training images shape: {X_train_resized.shape}")
#print(f"Resized training images shape: {X_val_resized.shape}")
#print(f"Resized validation images shape: {X_test_resized.shape}")

In [ ]:
# Uncomment below Data Augmentation
'''
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),  # flip horizontally
    tf.keras.layers.RandomRotation(0.2),       # 0.2 = rotate +/- 20% of 360 degress = +/- 72
    tf.keras.layers.RandomZoom(0.2)            # 0.2 = 80%-120% of original size
])

train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
'''

In [ ]:
X_train_processed = tf.keras.applications.mobilenet_v2.preprocess_input(X_train)
X_val_processed = tf.keras.applications.mobilenet_v2.preprocess_input(X_val)
X_test_processed = tf.keras.applications.mobilenet_v2.preprocess_input(X_test)

In [ ]:
# stops model training when the validation performance stops improving.
earlystopping = tf.keras.callbacks.EarlyStopping(patience=2,          # how many epochs the model waits before stopping.
                                                 monitor="val_loss",  # After every epoch, the model checks validation loss
                                                 restore_best_weights=True  # the model restores the weights from the best epoch.
                                                 )
history_model = final_model.fit(X_train_processed, y_train,
                                epochs=10,                # Maximum number of training cycles.
                                batch_size=BATCH_SIZE,    # #training images that the model processes before updating its weights once.
                                validation_data=(X_val_processed, y_val),   # Use this dataset to compute validation loss
                                callbacks=[earlystopping]
                                )


In [ ]:
final_model.save(DATA_NAME_PATH + 'transfer_learning_model.keras')

### Performance Evaluation

In [ ]:
# Plot the learning curves
import matplotlib.pyplot as plt
plt.figure(figsize=(15,5))
plt.subplot(121)
plt.subplot(121)
try:
    plt.plot(history_model.history['accuracy'])
    plt.plot(history_model.history['val_accuracy'])
except KeyError:
    plt.plot(history_model.history['acc'])
    plt.plot(history_model.history['val_acc'])
plt.title('Accuracy vs. epochs')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Training', 'Validation'], loc='lower right')
plt.subplot(122)
plt.plot(history_model.history['loss'])
plt.plot(history_model.history['val_loss'])
plt.title('Loss vs. epochs')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Training', 'Validation'], loc='upper right')
plt.show()

In [ ]:
new_model_test_loss, new_model_test_acc = final_model.evaluate(X_test_processed, y_test, verbose=0)
print("Test loss: {}".format(new_model_test_loss))
print("Test accuracy: {}".format(new_model_test_acc))

In [ ]:
predictions = final_model.predict(X_test_processed)
predictions

In [ ]:
y_predicted = np.argmax(predictions, axis=1)
y_predicted

In [ ]:
predicted_classes = [class_names[x] for x in y_predicted]
true_classes = [class_names[x] for x in y_test]

In [ ]:
import pandas as pd

results_df = pd.DataFrame({
    "Actual": true_classes,
    "Predicted": predicted_classes,
})
results_df.head()

In [ ]:
results_df["Correct"] = results_df["Actual"] == results_df["Predicted"]
num_correct = results_df["Correct"].sum()
print("Total Predictons: ", len(results_df), ", Correct predictions:", num_correct, ' Correct Percent: ', num_correct/len(results_df))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(true_classes, predicted_classes)
print(cm)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_predicted, target_names=class_names))

In [ ]:
class_names = np.array(class_list)

plt.figure(figsize=(15,10))

# Random sample of test images
inx = np.random.choice(X_test.shape[0], 15, replace=False)

for n, i in enumerate(inx):
    ax = plt.subplot(3,5,n+1)
    img = (X_test[i] + 1) / 2
    plt.imshow(img*255)
    #plt.imshow(X_test[i] * 255)

    true_label = true_classes[i]
    pred_label = predicted_classes[i]

    plt.title(f"True: {true_label}\nPred: {pred_label}")
    plt.axis('off')

plt.tight_layout()